In [7]:
from samap.mapping import SAMAP
from samap.analysis import get_mapping_scores
from samap.utils import save_samap, load_samap
from samalg import SAM

import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import networkx as nx
import anndata as ad

from collections import defaultdict
import itertools, os, gc, pickle

## Config

In [8]:
LEVEL     = 'ss_subclass_v5_nounlabeled_nn'      # input cell-type label
OUT_LEVEL = 'ss_subclass_nounlabeled_nmm_v5_nn'  # consensus column written back
THRESH    = 0.2                                # SAMap score floor
DATE      = '09072026'

SAM_DIR   = '../../Active_SAM_joined/'
SAMAP_DIR = '../../Active_SAMap_Joined/active_samap/Non-mammal/'
BLOCKS    = f'nonmam_samap_blocks_{DATE}.pkl'    # cached mapping-table blocks

H5AD = {
    'cj': SAM_DIR + 'SAM_CJ_joined_v2_cleaned_03122025.h5ad',
    'ac': SAM_DIR + 'SAM_AC_ncbi_soupx_cleaned_03122025.h5ad',
    'xt': SAM_DIR + 'SAM_XT_joined_Slc17a6_cleaned_03122205_nostale.h5ad',
    'dr': SAM_DIR + 'SAM_DR_ncbi_joined_cleaned_07172026.h5ad',
}

# v5 rebuild of the pairwise SAMap runs (09032026 / 09052026); xt-dr is unchanged.
PAIRS = {
    ('cj', 'ac'): SAMAP_DIR + 'sm_cj_cleaned_ac_ncbi_cleaned_09032026.pkl',
    ('cj', 'xt'): SAMAP_DIR + 'sm_cj_cleaned_xt_cleaned_09032026.pkl',
    ('cj', 'dr'): SAMAP_DIR + 'sm_cj_cleaned_dr_ncbi_cleaned_09032026.pkl',
    ('ac', 'xt'): SAMAP_DIR + 'sm_ac_ncbi_cleaned_xt_cleaned_09052026.pkl',
    ('ac', 'dr'): SAMAP_DIR + 'sm_ac_ncbi_cleaned_dr_ncbi_cleaned_09052026.pkl',
    ('xt', 'dr'): SAMAP_DIR + 'sm_xt_cleaned_dr_cleaned_07172026.pkl',
}

SPECIES      = ['cj', 'ac', 'xt', 'dr']   # also the prefix order used in group names
PREFIX_COLOR = {'cj': 'red', 'ac': 'blue', 'xt': 'green', 'dr': 'purple'}

print('cached blocks:', 'found' if os.path.exists(BLOCKS) else 'absent (section 2 will rebuild)')
for f in list(H5AD.values()) + list(PAIRS.values()):
    print(('OK   ' if os.path.exists(f) else 'MISS '), f)

cached blocks: absent (section 2 will rebuild)
OK    ../../Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad
OK    ../../Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad
OK    ../../Active_SAM_joined/SAM_XT_joined_Slc17a6_cleaned_03122205_nostale.h5ad
OK    ../../Active_SAM_joined/SAM_DR_ncbi_joined_cleaned_07172026.h5ad
OK    ../../Active_SAMap_Joined/active_samap/Non-mammal/sm_cj_cleaned_ac_ncbi_cleaned_09032026.pkl
OK    ../../Active_SAMap_Joined/active_samap/Non-mammal/sm_cj_cleaned_xt_cleaned_09032026.pkl
OK    ../../Active_SAMap_Joined/active_samap/Non-mammal/sm_cj_cleaned_dr_ncbi_cleaned_09032026.pkl
OK    ../../Active_SAMap_Joined/active_samap/Non-mammal/sm_ac_ncbi_cleaned_xt_cleaned_09052026.pkl
OK    ../../Active_SAMap_Joined/active_samap/Non-mammal/sm_ac_ncbi_cleaned_dr_ncbi_cleaned_09052026.pkl
OK    ../../Active_SAMap_Joined/active_samap/Non-mammal/sm_xt_cleaned_dr_cleaned_07172026.pkl


In [9]:
# The pickles predate LEVEL, so the labels are read from the h5ads here and injected
# into the pickled SAM objects in section 2. Barcodes / gene names are kept alongside
# so identity of the pickled SAM against the h5ad can be checked before that injection.
labels    = {}   # species -> Series of LEVEL indexed by barcode
var_names = {}   # species -> Index of gene names

for s in SPECIES:
    a = ad.read_h5ad(H5AD[s], backed='r')
    if LEVEL not in a.obs.columns:
        raise KeyError(f'{s}: {LEVEL} missing from {os.path.basename(H5AD[s])}')
    labels[s]    = a.obs[LEVEL].astype(str)
    var_names[s] = a.var_names.copy()
    print(f'{s}: {a.n_obs} cells, {a.n_vars} genes, {labels[s].nunique()} distinct labels')
    del a

cj: 74758 cells, 20032 genes, 137 distinct labels
ac: 47900 cells, 23671 genes, 136 distinct labels
xt: 42221 cells, 19621 genes, 125 distinct labels
dr: 61430 cells, 25503 genes, 59 distinct labels


In [10]:
def is_neuronal(item):
    """Non-neuronal subclasses are suffixed NN, e.g. '326 OPC NN'."""
    return not item.endswith('NN')


def is_species_cluster(item, prefix):
    """cj_1 -> True; cj_m066_m067 -> False; '128 VMH Fezf1 Glut' / 'Unlabeled' -> False"""
    return len(item) > 3 and item[:2] == prefix and item[2] == '_' and item[3] != 'm'


node_species = {}
for s in SPECIES:
    keep = [i for i in labels[s].unique() if is_species_cluster(i, s)]
    node_species.update({i: s for i in keep})
    print(f'{s}: {len(keep)} neuronal nodes')
    print({i: s for i in keep})

cell_counts = {n: int(labels[node_species[n]].value_counts().get(n, 0)) for n in node_species}
print('\ntotal nodes:', len(node_species))

cj: 32 neuronal nodes
{'cj_3': 'cj', 'cj_9': 'cj', 'cj_8': 'cj', 'cj_18': 'cj', 'cj_16': 'cj', 'cj_6': 'cj', 'cj_4': 'cj', 'cj_5': 'cj', 'cj_23': 'cj', 'cj_24': 'cj', 'cj_1': 'cj', 'cj_12': 'cj', 'cj_17': 'cj', 'cj_7': 'cj', 'cj_20': 'cj', 'cj_2': 'cj', 'cj_22': 'cj', 'cj_15': 'cj', 'cj_30': 'cj', 'cj_32': 'cj', 'cj_29': 'cj', 'cj_25': 'cj', 'cj_21': 'cj', 'cj_10': 'cj', 'cj_13': 'cj', 'cj_11': 'cj', 'cj_27': 'cj', 'cj_31': 'cj', 'cj_19': 'cj', 'cj_33': 'cj', 'cj_28': 'cj', 'cj_34': 'cj'}
ac: 34 neuronal nodes
{'ac_16': 'ac', 'ac_14': 'ac', 'ac_8': 'ac', 'ac_0': 'ac', 'ac_13': 'ac', 'ac_9': 'ac', 'ac_4': 'ac', 'ac_7': 'ac', 'ac_2': 'ac', 'ac_19': 'ac', 'ac_26': 'ac', 'ac_33': 'ac', 'ac_32': 'ac', 'ac_35': 'ac', 'ac_24': 'ac', 'ac_18': 'ac', 'ac_28': 'ac', 'ac_1': 'ac', 'ac_12': 'ac', 'ac_11': 'ac', 'ac_20': 'ac', 'ac_15': 'ac', 'ac_29': 'ac', 'ac_6': 'ac', 'ac_37': 'ac', 'ac_10': 'ac', 'ac_5': 'ac', 'ac_23': 'ac', 'ac_25': 'ac', 'ac_27': 'ac', 'ac_21': 'ac', 'ac_31': 'ac', 'ac_36': 'ac

## 2. Mapping-table blocks

One block per pair: rows = species `b` types, columns = species `a` types, restricted to graph
nodes. Cached to `BLOCKS` — the SAMap pickles are only loaded if the cache is missing.

Each pickle is checked against its h5ad (barcodes, genes, and an `X` probe) before `LEVEL` is
injected onto the pickled SAM objects, since the pickles predate the v5 labels.

In [11]:
def eligible(block):
    return block.loc[[i for i in block.index   if i[3:] in node_species],
                     [c for c in block.columns if c[3:] in node_species]]


def strip_prefix(idx, org):
    """SAMap prepends '<org>_' to the gene names of the SAM objects it holds, the
    same way it does to the mapping-table labels. Strip it so names line up with the
    h5ad, but only when every entry carries it. Barcodes are NOT prefixed, so this is
    applied to var_names only."""
    idx = pd.Index(idx)
    pre = org + '_'
    if len(idx) and idx.str.startswith(pre).all():
        return pd.Index([x[len(pre):] for x in idx]), True
    return idx, False


def check_sam_matches_h5ad(sm, org, n_probe=200):
    """Is sm.sams[org] the same object as H5AD[org]?

    Hard-fails only on the condition that would corrupt the label injection --
    cells in the pickle that do not exist in the h5ad. Gene-set and cell-count
    differences are reported but tolerated, since SAMap filters during
    preprocessing. Also compares X on a sample of shared cells to catch a pickle
    built from a different generation of the same filename.

    Gene names are compared with the SAMap '<org>_' prefix stripped; the pickle
    itself is indexed positionally, so its own naming never matters.
    """
    adata = sm.sams[org].adata
    flag  = lambda ok: 'OK  ' if ok else 'DIFF'

    pkl_bc  = pd.Index(adata.obs_names)          # barcodes are not prefixed
    h5_bc   = labels[org].index
    missing = pkl_bc.difference(h5_bc)
    shared  = pkl_bc.intersection(h5_bc)
    print(f'    [{org}] cells  pickle {len(pkl_bc)} / h5ad {len(h5_bc)}   '
          f'shared {len(shared)}   {flag(len(pkl_bc) == len(h5_bc) == len(shared))}')
    if len(missing):
        raise ValueError(
            f'{org}: {len(missing)} barcodes in the pickle are absent from '
            f'{os.path.basename(H5AD[org])} (e.g. {list(missing[:3])}). The pickle was '
            f'built from a different object -- labels cannot be transferred.')

    pkl_var, var_pre = strip_prefix(adata.var_names, org)
    shared_var = pkl_var.intersection(var_names[org])
    print(f'    [{org}] genes  pickle {len(pkl_var)} / h5ad {len(var_names[org])}   '
          f'shared {len(shared_var)}   {flag(len(pkl_var) == len(shared_var))}'
          + ('   (gene prefix stripped)' if var_pre else ''))
    if not var_pre:
        print(f'    [{org}] note: gene names carry no {org}_ prefix -- check this is expected')

    # X on a deterministic sample of shared cells/genes, positional on both sides
    probe_bc  = shared[:: max(1, len(shared) // n_probe)][:n_probe]
    probe_var = shared_var[:: max(1, len(shared_var) // n_probe)][:n_probe]
    if len(probe_bc) and len(probe_var):
        a = ad.read_h5ad(H5AD[org], backed='r')
        sub_h5 = a[np.sort(a.obs_names.get_indexer(probe_bc)), :].to_memory()[:, probe_var].X
        del a
        sub_pk = adata[np.sort(pkl_bc.get_indexer(probe_bc)),
                       np.sort(pkl_var.get_indexer(probe_var))].X
        tot_h5 = float(np.asarray(sub_h5.sum()))
        tot_pk = float(np.asarray(sub_pk.sum()))
        same   = np.isclose(tot_h5, tot_pk, rtol=1e-5)
        print(f'    [{org}] X sum over {len(probe_bc)}x{len(probe_var)} probe: '
              f'pickle {tot_pk:.4g} / h5ad {tot_h5:.4g}   {flag(same)}'
              + ('' if same else '   <-- different values or normalisation'))
        del sub_h5, sub_pk
    return len(missing) == 0


def inject_level(sm, org):
    """Put LEVEL onto the pickled SAM's obs, aligned by barcode."""
    adata = sm.sams[org].adata
    vals  = labels[org].reindex(pd.Index(adata.obs_names))
    n_missing = int(vals.isna().sum())
    adata.obs[LEVEL] = vals.fillna('Unlabeled').astype(str).values
    print(f'    [{org}] injected {LEVEL}: {adata.obs[LEVEL].nunique()} distinct'
          + (f', {n_missing} cells fell back to Unlabeled' if n_missing else ''))


def compute_blocks(PAIRS, LEVEL, check=True):
    """Load each SAMap object once and keep the node-restricted score block."""
    out = {}
    out_test = {}
    for (a, b), path in PAIRS.items():
        print(f'=== {a} vs {b} : {os.path.basename(path)}')
        sm = load_samap(path)
        for org in (a, b):
            if check:
                check_sam_matches_h5ad(sm, org)
            if LEVEL not in sm.sams[org].adata.obs.columns:
                inject_level(sm, org)

        D, MappingTable = get_mapping_scores(sm, {a: LEVEL, b: LEVEL})
        full = MappingTable.loc[[i for i in MappingTable.index   if i[:2] == b],
                                [i for i in MappingTable.columns if i[:2] == a]]
        out[(a, b)] = eligible(full)
        out_test[(a,b)] = full
        print(f'    {full.shape} -> eligible {out[(a, b)].shape}')

        del sm, D, MappingTable, full
        gc.collect()
    return(out, out_test)


if os.path.exists(BLOCKS):
    with open(BLOCKS, 'rb') as fh:
        cached = pickle.load(fh)
    blocks = cached['eligible'] if isinstance(cached, dict) and 'eligible' in cached else cached
    print(f'loaded {len(blocks)} blocks from {BLOCKS}')
else:
    blocks, full_blocks = compute_blocks(PAIRS, LEVEL)
    with open(BLOCKS, 'wb') as fh:
        pickle.dump({'eligible': blocks}, fh)
    with open('test_full_' + BLOCKS, 'wb') as fh:
        pickle.dump({'eligible': full_blocks}, fh)
    print(f'saved {BLOCKS}')

for k, v in sorted(blocks.items()):
    print(f'  {k}: {v.shape}')

=== cj vs ac : sm_cj_cleaned_ac_ncbi_cleaned_09032026.pkl
    [cj] cells  pickle 74758 / h5ad 74758   shared 74758   OK  
    [cj] genes  pickle 20032 / h5ad 20032   shared 20032   OK     (gene prefix stripped)
    [cj] X sum over 200x200 probe: pickle 3814 / h5ad 3814   OK  
    [cj] injected ss_subclass_v5_nounlabeled_nn: 137 distinct
    [ac] cells  pickle 47900 / h5ad 47900   shared 47900   OK  
    [ac] genes  pickle 23671 / h5ad 23671   shared 23671   OK     (gene prefix stripped)
    [ac] X sum over 200x200 probe: pickle 2385 / h5ad 2385   OK  
    [ac] injected ss_subclass_v5_nounlabeled_nn: 136 distinct
    (136, 137) -> eligible (34, 32)
=== cj vs xt : sm_cj_cleaned_xt_cleaned_09032026.pkl
    [cj] cells  pickle 74758 / h5ad 74758   shared 74758   OK  
    [cj] genes  pickle 20032 / h5ad 20032   shared 20032   OK     (gene prefix stripped)
    [cj] X sum over 200x200 probe: pickle 3814 / h5ad 3814   OK  
    [cj] injected ss_subclass_v5_nounlabeled_nn: 137 distinct
    [xt] c

In [12]:
blocks[('cj', 'ac')]

,cj_cj_1,cj_cj_10,cj_cj_11,cj_cj_12,cj_cj_13,cj_cj_15,cj_cj_16,cj_cj_17,cj_cj_18,cj_cj_19,...,cj_cj_31,cj_cj_32,cj_cj_33,cj_cj_34,cj_cj_4,cj_cj_5,cj_cj_6,cj_cj_7,cj_cj_8,cj_cj_9
ac_ac_0,0.197064,0.000006,0.005984,0.000178,0.023929,0.006359,0.002829,0.034460,0.089462,0.000398,...,0.000000,0.000000,0.0,0.0,0.001248,0.002722,0.127734,0.040857,0.001212,0.003278
ac_ac_1,0.000010,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.026725,0.000000,0.000000,0.000000,0.000000,0.000012
ac_ac_10,0.062261,0.000000,0.001721,0.000000,0.002308,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000078,0.000030,0.000567,0.000000,0.000000,0.000000
ac_ac_11,0.000044,0.000000,0.000000,0.000000,0.000000,0.000000,0.004327,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.043591,0.000948,0.000050,0.000147,0.000049,0.000000
ac_ac_12,0.000000,0.443949,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000329,0.000546,0.000000,0.000000,0.000000,0.000000
ac_ac_13,0.000000,0.000000,0.000121,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.002472,0.0,0.0,0.000000,0.000000,0.000016,0.000000,0.000000,0.000000
ac_ac_14,0.003207,0.000000,0.000000,0.004636,0.000000,0.000000,0.000187,0.000000,0.002261,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.002528,0.000035,0.000000,0.000000,0.000000
ac_ac_15,0.000133,0.000000,0.826430,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.003612,...,0.000000,0.000000,0.0,0.0,0.000122,0.000000,0.000000,0.000000,0.000000,0.000000
ac_ac_16,0.032437,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000098,0.000109,0.000000,...,0.000000,0.000000,0.0,0.0,0.000000,0.000026,0.004347,0.000150,0.000000,0.000067
ac_ac_18,0.000082,0.000000,0.000000,0.009389,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.0,0.0,0.000599,0.000188,0.000000,0.000227,0.000149,0.000000


## 3. Best hits among eligible partners

`idxmax` over the node-only block, so every recorded partner is one that can form an edge.

In [13]:
def best_hits_from(block):
    out = {}
    for col in block.columns:
        item = col[3:]
        if block[col].isna().all() or block[col].max() == 0:
            out[item] = [None, 0.0]
            continue
        best = block[col].idxmax()
        out[item] = [best[3:], float(block.loc[best, col])]
    return out


mapping_frames = {}
for (a, b), blk in blocks.items():
    for (src, dst), d in (((a, b), best_hits_from(blk)), ((b, a), best_hits_from(blk.T))):
        df = pd.DataFrame.from_dict(d, orient='index',
                                    columns=['SAMAP match', 'SAMAP score'])
        df.index.name = 'source'
        fn = f'{src}_{dst}_nonmam_mapping_SAMap_{DATE}.csv'
        df.to_csv(fn)
        mapping_frames[(src, dst)] = df
        print(f'{fn}  ({len(df)} types, '
              f'{int((df["SAMAP score"] >= THRESH).sum())} at or above {THRESH})')

cj_ac_nonmam_mapping_SAMap_09072026.csv  (32 types, 11 at or above 0.2)
ac_cj_nonmam_mapping_SAMap_09072026.csv  (34 types, 14 at or above 0.2)
cj_xt_nonmam_mapping_SAMap_09072026.csv  (32 types, 12 at or above 0.2)
xt_cj_nonmam_mapping_SAMap_09072026.csv  (38 types, 10 at or above 0.2)
cj_dr_nonmam_mapping_SAMap_09072026.csv  (32 types, 9 at or above 0.2)
dr_cj_nonmam_mapping_SAMap_09072026.csv  (27 types, 3 at or above 0.2)
ac_xt_nonmam_mapping_SAMap_09072026.csv  (34 types, 16 at or above 0.2)
xt_ac_nonmam_mapping_SAMap_09072026.csv  (38 types, 19 at or above 0.2)
ac_dr_nonmam_mapping_SAMap_09072026.csv  (34 types, 9 at or above 0.2)
dr_ac_nonmam_mapping_SAMap_09072026.csv  (27 types, 7 at or above 0.2)
xt_dr_nonmam_mapping_SAMap_09072026.csv  (38 types, 9 at or above 0.2)
dr_xt_nonmam_mapping_SAMap_09072026.csv  (27 types, 8 at or above 0.2)


## 4. Edges: reciprocal best hit ≥ THRESH

In [14]:
G = nx.Graph()
G.add_nodes_from(node_species)

for (a, b) in PAIRS:
    for (src, dst) in ((a, b), (b, a)):
        fwd, rev = mapping_frames[(src, dst)], mapping_frames[(dst, src)]
        for u, row in fwd.iterrows():
            v = row['SAMAP match']
            score = float(row['SAMAP score'])
            if v in rev.index:
                if rev.loc[v, 'SAMAP match'] == u and score > THRESH:
                    G.add_edge(u, v, weight=score)

print(f'edges: {G.number_of_edges()}')

edges: 49


In [15]:
THRESH

0.2

In [16]:
pd.DataFrame([{'A': u, 'B': v, 'score': round(d['weight'], 3)}
               for u, v, d in G.edges(data=True)]).sort_values('score', ascending=False)

,A,B,score
33,ac_28,xt_37,0.903
32,ac_35,xt_30,0.902
18,cj_19,ac_23,0.826
17,cj_11,ac_15,0.826
15,cj_10,xt_25,0.644
26,ac_0,xt_10,0.636
7,cj_1,ac_9,0.617
0,cj_3,ac_0,0.613
5,cj_4,xt_26,0.585
8,cj_1,xt_0,0.579


## 5. Consensus groups

Connected components become homology groups, named for the species present in `cj, ac, xt, dr`
order plus a counter (`cj_ac_xt_1`). Types in no group keep their original label.

In [17]:
def name_groups(g):
    comps = [c for c in nx.connected_components(g) if len(c) > 1]
    counters = defaultdict(int)
    res, members = {}, {}
    for c in sorted(comps, key=len, reverse=True):
        combo = '_'.join(sorted({node_species[n] for n in c}, key=SPECIES.index))
        counters[combo] += 1
        name = f'{combo}_{counters[combo]}'
        members[name] = sorted(c)
        for n in c:
            res[n] = name
    return res, members


result, members = name_groups(G)
print(f'{len(members)} groups covering {len(result)} of {len(node_species)} types\n')

for name, mem in sorted(members.items(), key=lambda kv: -len(kv[1])):
    counts = {s: sum(1 for n in mem if node_species[n] == s) for s in SPECIES}
    dup = ', '.join(f'{s}x{counts[s]}' for s in SPECIES if counts[s] > 1)
    print(f'=== {name}   {len(mem)} types, {sum(cell_counts[n] for n in mem)} cells'
          + (f'   [DUPES: {dup}]' if dup else ''))
    for n in sorted(mem):
        nbrs = sorted(G[n].items(), key=lambda kv: -kv[1]['weight'])
        print(f"    {n:<8} {cell_counts[n]:>6} cells   -> "
              + ', '.join(f"{m}={d['weight']:.2f}" for m, d in nbrs))
    print()

19 groups covering 59 of 131 types

=== cj_ac_xt_dr_1   8 types, 9177 cells   [DUPES: cjx2, acx3, xtx2]
    ac_2        762 cells   -> xt_2=0.48
    ac_23        75 cells   -> cj_19=0.83, dr_0=0.30
    ac_4        633 cells   -> cj_2=0.50, xt_6=0.26
    cj_19       174 cells   -> ac_23=0.83, xt_6=0.56, dr_0=0.55
    cj_2        959 cells   -> ac_4=0.50, xt_2=0.34
    dr_0       5310 cells   -> cj_19=0.55, ac_23=0.30, xt_6=0.26
    xt_2        797 cells   -> ac_2=0.48, cj_2=0.34
    xt_6        467 cells   -> cj_19=0.56, ac_4=0.26, dr_0=0.26

=== cj_ac_xt_dr_2   5 types, 3965 cells   [DUPES: cjx2]
    ac_25        64 cells   -> xt_5=0.24, cj_8=0.21
    cj_16       224 cells   -> xt_5=0.47
    cj_8        398 cells   -> ac_25=0.21
    dr_6       2763 cells   -> xt_5=0.21
    xt_5        516 cells   -> cj_16=0.47, ac_25=0.24, dr_6=0.21

=== cj_ac_xt_dr_3   5 types, 4175 cells   [DUPES: xtx2]
    ac_6        572 cells   -> cj_4=0.45, xt_3=0.43, dr_7=0.31
    cj_4        570 cells   -> xt_2

In [18]:
summary = pd.DataFrame([
    {'group': name,
     'types': len(mem),
     'cells': sum(cell_counts[n] for n in mem),
     'species': len({node_species[n] for n in mem}),
     'dupes': ', '.join(f'{s}x{c}' for s, c in
                        ((s, sum(1 for n in mem if node_species[n] == s)) for s in SPECIES)
                        if c > 1),
     'min score': round(min(d['weight'] for _, _, d in G.subgraph(mem).edges(data=True)), 3),
     'members': ' '.join(mem)}
    for name, mem in members.items()]).sort_values(['species', 'types'], ascending=False)

pd.set_option('display.max_colwidth', 200)
summary.to_csv(f'nonmam_consensus_groups_SAMap_{DATE}.csv', index=False)
pd.Series(result, name='group').sort_index().to_frame().to_csv(
    f'nonmam_consensus_assignment_SAMap_{DATE}.csv')
summary.set_index('group')

,types,cells,species,dupes,min score,members
group,,,,,,
cj_ac_xt_dr_1,8,9177,4,"cjx2, acx3, xtx2",0.258,ac_2 ac_23 ac_4 cj_19 cj_2 dr_0 xt_2 xt_6
cj_ac_xt_dr_2,5,3965,4,cjx2,0.205,ac_25 cj_16 cj_8 dr_6 xt_5
cj_ac_xt_dr_3,5,4175,4,xtx2,0.314,ac_6 cj_4 dr_7 xt_26 xt_3
cj_ac_xt_dr_4,4,6557,4,,0.296,ac_9 cj_1 dr_1 xt_0
cj_ac_xt_dr_5,4,3616,4,,0.230,ac_12 cj_10 dr_5 xt_25
cj_ac_xt_1,3,3247,3,,0.264,ac_0 cj_3 xt_10
cj_xt_dr_1,3,3284,3,,0.319,cj_22 dr_4 xt_12
ac_xt_dr_1,3,1260,3,,0.361,ac_16 dr_17 xt_29
ac_xt_dr_2,3,1458,3,,0.222,ac_14 dr_19 xt_4


In [ ]:
def apply_labels(fn, s, dry_run=True):
    sam = SAM()
    sam.load_data(fn)

    src = sam.adata.obs[LEVEL].astype(str)
    fin = src.map(lambda x: result.get(x, x))

    print(f'{s}: {int((fin != src).sum())}/{len(src)} cells relabelled, '
          f'{src.nunique()} -> {fin.nunique()} distinct labels')
    print('    new labels:', sorted(set(fin) - set(src)))

    if dry_run:
        print('    dry run — nothing written')
        return sam

    sam.adata.obs[OUT_LEVEL] = pd.Categorical(fin.values)
    sam.save_anndata(fn)
    print(f'    saved {OUT_LEVEL} into {fn}')
    return sam

In [21]:
# Flip to dry_run=False to commit. Overwrites the four h5ads.
for s in SPECIES:
    apply_labels(H5AD[s], s, dry_run=False)
    gc.collect()

cj: 5076/74758 cells relabelled, 137 -> 135 distinct labels
    new labels: ['cj_ac_1', 'cj_ac_2', 'cj_ac_3', 'cj_ac_xt_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_2', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4', 'cj_ac_xt_dr_5', 'cj_xt_dr_1']
    saved ss_subclass_nounlabeled_nmm_v5_nn into ../../Active_SAM_joined/SAM_CJ_joined_v2_cleaned_03122025.h5ad
ac: 7723/47900 cells relabelled, 136 -> 134 distinct labels
    new labels: ['ac_dr_1', 'ac_xt_1', 'ac_xt_2', 'ac_xt_3', 'ac_xt_4', 'ac_xt_dr_1', 'ac_xt_dr_2', 'ac_xt_dr_3', 'cj_ac_1', 'cj_ac_2', 'cj_ac_3', 'cj_ac_xt_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_2', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4', 'cj_ac_xt_dr_5']
    saved ss_subclass_nounlabeled_nmm_v5_nn into ../../Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad
xt: 6151/42221 cells relabelled, 125 -> 123 distinct labels
    new labels: ['ac_xt_1', 'ac_xt_2', 'ac_xt_3', 'ac_xt_4', 'ac_xt_dr_1', 'ac_xt_dr_2', 'ac_xt_dr_3', 'cj_ac_xt_1', 'cj_ac_xt_dr_1', 'cj_ac_xt_dr_2', 'cj_ac_xt_dr_3', 'cj_ac_xt_dr_4', '